# Namma Connect V2 — Backend Manual Testing & QA Console

This notebook is an interactive, comprehensive, manual backend testing and QA console for **Namma Connect V2**.
It is designed to inspect, execute, and verify all backend services, endpoints, schemas, authorization boundaries, and workflows against the actual implementation without modifying production business logic.

---

### Architecture & Capabilities Tested
- **FastAPI Core & Routing**: `/health`, `/api/v2/*`
- **Authentication & RBAC**: Customer, Partner/Farmer, Creator, Admin roles, JWT lifecycle, password workflows
- **Marketplace Catalog & Availability**: Dynamic calendar slots, filters, ratings, reviews
- **Booking Engine & Authoritative Calculations**: Capacity, pricing immutability, reservation state machine
- **Payments & Cryptographic Verification**: Razorpay order flow, HMAC-SHA256 signatures, idempotency, anti-tampering
- **Cancellation & Refunds**: Policy-based refund calculation, isolation, status tracking
- **Partner Operations**: Listing management, moderation submission, earnings, payout ledger
- **Creator Hub & Collaborations**: Profiles, portfolios, packages, multi-party proposals
- **Communication Systems**: Notifications, conversations, support ticketing
- **AI & Discovery**: Travel AI assistant, unified semantic search, Gemini embeddings pipeline
- **Dev Tools & Security Audit**: Database inspection, dev seed verification, negative attack surface testing


---
## Testing Mode Configuration

This notebook supports **two testing modes**:

1. **Mode A: In-Process Test Mode (DEFAULT & RECOMMENDED)**
   - Uses `fastapi.testclient.TestClient(app)` with an isolated SQLite in-memory database (`sqlite:///:memory:`).
   - Follows the exact architecture of `v2/backend/tests/conftest.py`.
   - Complete isolation: no external database or running server required.
   - All external services (Razorpay, Resend, Cloudinary, Gemini) are safely mocked by default.

2. **Mode B: Live API Server Test Mode (OPTIONAL)**
   - Uses `httpx` or `requests` to test a running backend instance at `http://127.0.0.1:8000`.
   - Requires the FastAPI server to be launched separately (`uvicorn app.main:app`).

> **Note**: Only choose ONE mode at a time. The notebook defaults to **Mode A**.


---
# 0. Setup and Environment

This section imports backend dependencies, performs environment safety checks, sets up the isolated SQLite test database, and initializes the dual-mode client adapter and testing helpers.


In [ ]:
import os
import sys
import uuid
import hmac
import hashlib
import json
from datetime import datetime, date, timedelta
from typing import Dict, Any, Optional, List
from pathlib import Path

# Ensure UTF-8 output encoding across Windows shells & notebooks
if hasattr(sys.stdout, "reconfigure"):
    try:
        sys.stdout.reconfigure(encoding="utf-8")
    except Exception:
        pass

# Add v2/backend to sys.path
PROJECT_DIR = Path("c:/Users/13ver/Desktop/New folder/project/namma_connect")
BACKEND_DIR = PROJECT_DIR / "v2" / "backend"
if not BACKEND_DIR.exists():
    CURRENT_DIR = Path.cwd()
    BACKEND_DIR = CURRENT_DIR / "v2" / "backend"
    if not BACKEND_DIR.exists():
        if (CURRENT_DIR / "app").exists():
            BACKEND_DIR = CURRENT_DIR
        elif (CURRENT_DIR.parent / "v2" / "backend").exists():
            BACKEND_DIR = CURRENT_DIR.parent / "v2" / "backend"

sys.path.insert(0, str(BACKEND_DIR))
print(f"[Setup] Backend directory added to sys.path: {BACKEND_DIR}")


In [ ]:
# Environment and Safety Guard Inspection
from app.core.config import settings

print("=" * 70)
print(f"  {settings.PROJECT_NAME} V2 — Environment Inspection")
print("=" * 70)
print(f"  Version:         {settings.VERSION}")
print(f"  API V2 Prefix:   {settings.API_V2_PREFIX}")
print(f"  Environment:     {settings.ENV}")
print(f"  Debug Mode:      {settings.DEBUG}")

# CRITICAL SAFETY CHECK: Refuse execution if environment is production
assert settings.ENV.lower() != "production", (
    "FATAL SAFETY VIOLATION: Execution refused! Environment is set to 'production'."
)

# Display configured external integrations status without leaking secrets
svc_status = settings.get_configured_services()
print("\n[External Integrations Status (Configuration flags only, no secrets leaked)]:")
print(f"  PostgreSQL:      {'CONFIGURED' if svc_status.get('postgresql') else 'NOT CONFIGURED'}")
print(f"  Redis:           {'CONFIGURED' if svc_status.get('redis') else 'NOT CONFIGURED'}")
print(f"  Razorpay:        {'CONFIGURED' if svc_status.get('razorpay') else 'NOT CONFIGURED (Mocked)'}")
print(f"  Cloudinary:      {'CONFIGURED' if svc_status.get('cloudinary') else 'NOT CONFIGURED (Mocked)'}")
print(f"  Gemini:          {'CONFIGURED' if svc_status.get('gemini') else 'NOT CONFIGURED (Deterministic fallback)'}")
print(f"  Resend Email:    {'CONFIGURED' if svc_status.get('resend') else 'NOT CONFIGURED (Mocked)'}")
print(f"  Google OAuth:    {'CONFIGURED' if svc_status.get('google_oauth') else 'NOT CONFIGURED (Mocked)'}")
print("=" * 70)


In [ ]:
# Database Setup (Isolated SQLite In-Memory) and Dependency Override
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker, Session
from sqlalchemy.pool import StaticPool
from fastapi.testclient import TestClient

from app.main import app as fastapi_app
from app.core.database import Base, get_db
import app.models  # Register all SQLAlchemy models
from app.services.marketplace import MarketplaceService

# 1. Create in-memory SQLite engine with StaticPool
TEST_DATABASE_URL = "sqlite:///:memory:"
test_engine = create_engine(
    TEST_DATABASE_URL,
    connect_args={"check_same_thread": False},
    poolclass=StaticPool,
)
TestingSessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=test_engine)

# 2. Create all tables from models
Base.metadata.create_all(bind=test_engine)
print("[Database] Isolated in-memory SQLite tables created successfully.")

# 3. Create persistent test session
test_db: Session = TestingSessionLocal()

# Pre-seed verified catalog listings
MarketplaceService.ensure_seeded(test_db)
print(f"[Database] Default verified catalog seeded. Active services count: {test_db.query(app.models.Service).count()}")

# 4. Dependency override for FastAPI get_db
def override_get_db():
    try:
        yield test_db
    finally:
        pass

fastapi_app.dependency_overrides[get_db] = override_get_db


In [ ]:
# Client Mode Configuration & Display Helpers
TEST_MODE = "IN_PROCESS"  # Change to "LIVE_SERVER" if testing against http://127.0.0.1:8000
LIVE_SERVER_URL = "http://127.0.0.1:8000"

in_process_client = TestClient(fastapi_app)

class TestClientWrapper:
    """Unified client adapter supporting both in-process and live server testing."""
    def __init__(self, mode="IN_PROCESS", base_url=""):
        self.mode = mode
        self.base_url = base_url.rstrip("/")
        self.local_client = in_process_client

    def request(self, method, url, **kwargs):
        if self.mode == "IN_PROCESS":
            return self.local_client.request(method, url, **kwargs)
        else:
            import httpx
            full_url = f"{self.base_url}{url}"
            return httpx.request(method, full_url, **kwargs)

    def get(self, url, **kwargs): return self.request("GET", url, **kwargs)
    def post(self, url, **kwargs): return self.request("POST", url, **kwargs)
    def put(self, url, **kwargs): return self.request("PUT", url, **kwargs)
    def patch(self, url, **kwargs): return self.request("PATCH", url, **kwargs)
    def delete(self, url, **kwargs): return self.request("DELETE", url, **kwargs)

client = TestClientWrapper(mode=TEST_MODE, base_url=LIVE_SERVER_URL)
print(f"[Client] TestClient initialized in '{TEST_MODE}' mode.")


In [ ]:
# Test Result Collector & QA Display Utilities
class TestResultCollector:
    """Lightweight test result aggregator to compile the final summary table."""
    def __init__(self):
        self.records = {}

    def log(self, section: str, passed: bool, notes: str = ""):
        if section not in self.records:
            self.records[section] = {"tested": 0, "passed": 0, "failed": 0, "notes": []}
        self.records[section]["tested"] += 1
        if passed:
            self.records[section]["passed"] += 1
        else:
            self.records[section]["failed"] += 1
        if notes:
            self.records[section]["notes"].append(notes)

    def summary(self):
        print(f"| {'Area / Section':<32} | {'Tested':>6} | {'Passed':>6} | {'Failed':>6} | {'Notes':<30} |")
        print(f"|{'-'*34}|{'-'*8}:|{'-'*8}:|{'-'*8}:|{'-'*32}|")
        for sec, data in self.records.items():
            notes_str = "; ".join(data["notes"][:2])
            print(f"| {sec:<32} | {data['tested']:>6} | {data['passed']:>6} | {data['failed']:>6} | {notes_str:<30} |")

collector = TestResultCollector()

def mask_token(token: Optional[str]) -> Optional[str]:
    """Safely mask tokens to prevent full credential leaks in logs."""
    if not token:
        return None
    return token[:14] + "..." + token[-6:] if len(token) > 20 else "***"

def show_response(response, title: str = ""):
    """Standardized formatted printer for API responses."""
    if title:
        print(f"=== [ {title} ] ===")
    print(f"HTTP Status: {response.status_code}")
    print(f"Endpoint:    {getattr(response.request, 'method', 'REQ')} {getattr(response.request, 'url', '')}")
    try:
        data = response.json()
        print("Response JSON:")
        print(json.dumps(data, indent=2, default=str))
        return data
    except Exception:
        print(f"Response Body: {response.text}")
        return response.text

def assert_status(response, expected_status: int, section_name: str = "", note: str = ""):
    """Assert response status code with friendly debug logging."""
    is_ok = (response.status_code == expected_status)
    if not is_ok:
        print(f"\n[FAIL] Expected HTTP {expected_status}, got {response.status_code}")
        print(f"Body: {response.text}")
    if section_name:
        collector.log(section_name, is_ok, note)
    assert is_ok, f"Expected HTTP {expected_status}, got {response.status_code}: {response.text}"

def auth_headers(token: str) -> Dict[str, str]:
    """Construct Authorization Bearer headers."""
    return {"Authorization": f"Bearer {token}"}

# Global test identity registry
test_state = {
    "suffix": uuid.uuid4().hex[:6],
    "customer": {},
    "partner": {},
    "farmer": {},
    "creator": {},
    "admin": {},
    "service": {},
    "booking": {},
    "payment": {},
    "refund": {},
    "payout": {},
}
print(f"[Setup] Test suite initialized with dynamic run suffix: '{test_state['suffix']}'")


---
# 1. Health and API Smoke Tests

Verify basic operational status across root `/health` and sub-module status gateways.


In [ ]:
# Test Root Health Gateway
res_health = client.get("/health")
data_health = show_response(res_health, title="Root Health Gateway")
assert_status(res_health, 200, "Health", "Root health check ok")
assert data_health["status"] in ["healthy", "ok"]
assert data_health["environment"] == settings.ENV
print("[PASS] Interpretation: Root health check responded with status healthy and correct environment.")


In [ ]:
# Test API V2 Auth, Search, Services, Bookings, Admin Gateways & OpenAPI
endpoints_to_smoke = [
    ("/api/v2/auth/status", 200, "Auth Status Gateway"),
    ("/api/v2/search?q=coffee", 200, "Search Smoke Query"),
    ("/api/v2/services", 200, "Marketplace Services List"),
    ("/api/v2/bookings", 200, "Bookings Status Gateway"),
    ("/api/v2/admin/status", 200, "Admin Status Gateway"),
    ("/api/v2/openapi.json", 200, "OpenAPI Specification JSON"),
    ("/api/v2/docs", 200, "Swagger UI Documentation"),
]

for url, expected, label in endpoints_to_smoke:
    resp = client.get(url)
    assert_status(resp, expected, "Health", label)
    print(f"[OK] {label:<28} -> HTTP {resp.status_code}")

print("\n[PASS] Interpretation: All core API v2 smoke gateways and documentation endpoints are reachable.")


---
# 2. Authentication

Test the full authentication lifecycle: registration for multiple roles, duplicate prevention, login validations, `/auth/me`, token refreshing, password workflows, and logout.


In [ ]:
# Register Test Identities (Customer, Partner, Farmer, Creator, Admin)
suffix = test_state["suffix"]
roles_to_register = [
    ("customer", f"cust_{suffix}@nammaconnect.in", "Priya Sharma", "+919876540001"),
    ("partner", f"host_{suffix}@nammaconnect.in", "Bopaiah Kodagu", "+919876540002"),
    ("creator", f"creator_{suffix}@nammaconnect.in", "Ananya Visuals", "+919876540003"),
    ("admin", f"admin_{suffix}@nammaconnect.in", "Platform Admin", "+919876540004"),
]

for role, email, name, phone in roles_to_register:
    payload = {
        "email": email,
        "password": "SecurePassword123!",
        "full_name": name,
        "mobile": phone,
        "role": role,
    }
    resp = client.post("/api/v2/auth/register", json=payload)
    assert_status(resp, 201, "Authentication", f"Register {role}")
    data = resp.json()
    assert "access_token" in data
    assert "refresh_token" in data
    assert data["user"]["role"] == role
    assert "password" not in data["user"]
    assert "hashed_password" not in data["user"]
    
    test_state[role] = {
        "email": email,
        "password": "SecurePassword123!",
        "token": data["access_token"],
        "refresh_token": data["refresh_token"],
        "id": data["user"]["id"],
    }
    print(f"[OK] Registered {role.upper()}: {email} (Token: {mask_token(data['access_token'])})")

print("\n[PASS] Interpretation: Successfully registered isolated identities across customer, partner, creator, and admin roles.")


In [ ]:
# Test Duplicate Registration Rejection
dup_payload = {
    "email": test_state["customer"]["email"],
    "password": "AnotherPassword123!",
    "full_name": "Duplicate User",
    "role": "customer",
}
resp_dup = client.post("/api/v2/auth/register", json=dup_payload)
show_response(resp_dup, title="Duplicate Registration Rejection")
assert_status(resp_dup, 400, "Authentication", "Duplicate email rejection")
assert "already exists" in resp_dup.json()["detail"].lower()
print("[PASS] Interpretation: System rejected duplicate user registration with 400 Bad Request.")


In [ ]:
# Test Login Workflows (Valid, Invalid Password, Unknown User)
# 1. Valid Login
resp_valid_login = client.post(
    "/api/v2/auth/login",
    json={"email": test_state["customer"]["email"], "password": "SecurePassword123!"}
)
assert_status(resp_valid_login, 200, "Authentication", "Valid login")
assert "access_token" in resp_valid_login.json()

# 2. Invalid Password
resp_bad_pwd = client.post(
    "/api/v2/auth/login",
    json={"email": test_state["customer"]["email"], "password": "WrongPassword!"}
)
assert_status(resp_bad_pwd, 401, "Authentication", "Invalid password rejection")

# 3. Unknown User
resp_unknown = client.post(
    "/api/v2/auth/login",
    json={"email": f"unknown_{suffix}@example.com", "password": "Password123!"}
)
assert_status(resp_unknown, 401, "Authentication", "Unknown user login rejection")
print("[PASS] Interpretation: Login endpoint correctly verifies credentials and rejects invalid attempts with 401.")


In [ ]:
# Test Current User Profile (/auth/me) & Refresh Token Flow
# 1. /auth/me with Bearer token
resp_me = client.get(
    "/api/v2/auth/me",
    headers=auth_headers(test_state["customer"]["token"])
)
assert_status(resp_me, 200, "Authentication", "Get current user profile")
data_me = resp_me.json()
assert data_me["email"] == test_state["customer"]["email"]
assert data_me["role"] == "customer"

# 2. /auth/me unauthenticated -> 401
resp_me_unauth = client.get("/api/v2/auth/me")
assert_status(resp_me_unauth, 401, "Authentication", "Unauthenticated /me rejection")

# 3. Refresh Token Flow
resp_refresh = client.post(
    "/api/v2/auth/refresh",
    json={"refresh_token": test_state["customer"]["refresh_token"]}
)
assert_status(resp_refresh, 200, "Authentication", "Token refresh")
assert "access_token" in resp_refresh.json()

# 4. Logout confirmation
resp_logout = client.post("/api/v2/auth/logout")
assert_status(resp_logout, 200, "Authentication", "Logout acknowledgement")
print("[PASS] Interpretation: Authenticated sessions, profile retrieval, refresh tokens, and logout confirmed.")


In [ ]:
# Test Password Workflows (Forgot Password, Change Password)
# 1. Forgot Password (Generic response, does not leak email existence)
resp_forgot = client.post(
    "/api/v2/auth/forgot-password",
    json={"email": test_state["customer"]["email"]}
)
assert_status(resp_forgot, 200, "Authentication", "Forgot password generic response")
assert resp_forgot.json()["success"] is True

# 2. Change Password
resp_change = client.post(
    "/api/v2/auth/change-password",
    json={
        "current_password": "SecurePassword123!",
        "new_password": "NewSecurePassword123!",
    },
    headers=auth_headers(test_state["customer"]["token"])
)
assert_status(resp_change, 200, "Authentication", "Change password")

# Revert password back for subsequent tests
client.post(
    "/api/v2/auth/change-password",
    json={
        "current_password": "NewSecurePassword123!",
        "new_password": "SecurePassword123!",
    },
    headers=auth_headers(test_state["customer"]["token"])
)
print("[PASS] Interpretation: Password reset and change password workflows verified successfully.")


---
# 3. Authentication and RBAC

Verify role-based access control (RBAC) boundaries across anonymous, customer, partner, creator, and admin identities.


In [ ]:
# RBAC Boundary Enforcement Matrix
cust_tok = test_state["customer"]["token"]
ptnr_tok = test_state["partner"]["token"]
adm_tok = test_state["admin"]["token"]

# 1. Customer attempts Partner endpoint (POST /api/v2/services) -> 403 Forbidden
res_c_on_p = client.post("/api/v2/services", headers=auth_headers(cust_tok))
assert_status(res_c_on_p, 403, "RBAC", "Customer blocked from partner service creation")

# 2. Customer attempts Admin endpoint (GET /api/v2/admin/overview) -> 403 Forbidden
res_c_on_a = client.get("/api/v2/admin/overview", headers=auth_headers(cust_tok))
assert_status(res_c_on_a, 403, "RBAC", "Customer blocked from admin overview")

# 3. Partner attempts Admin endpoint (GET /api/v2/admin/overview) -> 403 Forbidden
res_p_on_a = client.get("/api/v2/admin/overview", headers=auth_headers(ptnr_tok))
assert_status(res_p_on_a, 403, "RBAC", "Partner blocked from admin overview")

# 4. Partner accesses Partner endpoint -> 200 OK
res_p_on_p = client.get("/api/v2/services/partner/me", headers=auth_headers(ptnr_tok))
assert_status(res_p_on_p, 200, "RBAC", "Partner allowed partner me endpoint")

# 5. Admin accesses Admin endpoint -> 200 OK
res_a_on_a = client.get("/api/v2/admin/overview", headers=auth_headers(adm_tok))
assert_status(res_a_on_a, 200, "RBAC", "Admin allowed admin overview")

print("[PASS] Interpretation: RBAC boundaries strictly enforced. Privilege escalation attempts properly rejected.")


---
# 4. Services / Marketplace

Test marketplace discovery, filtering, pagination, and detailed service information.


In [ ]:
# Test Marketplace Listing with Filters
resp_services = client.get("/api/v2/services?limit=5")
data_services = show_response(resp_services, title="Marketplace Catalog Listing")
assert_status(resp_services, 200, "Marketplace", "List services")
services_list = data_services["data"]["services"]
assert len(services_list) > 0, "Expected at least one seeded service listing"

first_service = services_list[0]
test_state["service"]["id"] = first_service["id"]
test_state["service"]["title"] = first_service["title"]
test_state["service"]["price"] = first_service["price"]

print(f"\nExtracted First Service:")
print(f"  ID:       {first_service['id']}")
print(f"  Title:    {first_service['title']}")
print(f"  Category: {first_service['category']}")
print(f"  Location: {first_service['location']}")
print(f"  Price:    INR {first_service['price']} / {first_service['unit']}")
print(f"  Status:   {first_service['status']}")


In [ ]:
# Test Service Detail, Reviews, and Availability endpoints
srv_id = test_state["service"]["id"]

# 1. Detail (returns ServiceDetailResponse with 'service' and 'reviews')
resp_detail = client.get(f"/api/v2/services/{srv_id}")
assert_status(resp_detail, 200, "Marketplace", "Get service detail")
detail_data = resp_detail.json()["data"]
assert detail_data["service"]["id"] == srv_id

# 2. Reviews
resp_reviews = client.get(f"/api/v2/services/{srv_id}/reviews")
assert_status(resp_reviews, 200, "Marketplace", "Get service reviews")

# 3. Availability
resp_avail = client.get(f"/api/v2/services/{srv_id}/availability")
assert_status(resp_avail, 200, "Marketplace", "Get service availability")
assert "days" in resp_avail.json()["data"]
print("[PASS] Interpretation: Public marketplace endpoints return complete service details, reviews, and availability matrix.")


---
# 5. Partner Service Management

Test partner service creation, editing, status submission, and cross-partner isolation.


In [ ]:
# Partner creates new draft service
ptnr_tok = test_state["partner"]["token"]
srv_payload = {
    "title": "Malnad Organic Spice Plantation Walk",
    "description": "Guided walking tour through heritage cardamon and pepper vines.",
    "category": "Experiences",
    "category_slug": "experiences",
    "location": "Sirsi, Uttara Kannada",
    "district": "Uttara Kannada",
    "state": "Karnataka",
    "price": 1200.0,
    "unit": "person",
    "max_capacity": 15,
    "duration_hours": 3.0,
    "inclusions": ["Guided Walk", "Herbal Tea", "Spice Sampler"],
    "amenities": ["Parking", "Restrooms", "Drinking Water"]
}

resp_create_srv = client.post(
    "/api/v2/services",
    json=srv_payload,
    headers=auth_headers(ptnr_tok)
)
data_create_srv = show_response(resp_create_srv, title="Partner Service Creation")
assert_status(resp_create_srv, 200, "Partner Services", "Partner creates service")
created_srv = data_create_srv["data"]
test_state["partner"]["service_id"] = created_srv["id"]
assert created_srv["status"] == "PENDING"
assert created_srv["price"] == 1200.0
print(f"[OK] Created partner draft service with ID: {created_srv['id']} (Status: {created_srv['status']})")


In [ ]:
# Partner retrieves and updates own service
partner_srv_id = test_state["partner"]["service_id"]

# 1. Partner Me
resp_p_me = client.get("/api/v2/services/partner/me", headers=auth_headers(ptnr_tok))
assert_status(resp_p_me, 200, "Partner Services", "List partner own services")
assert any(s["id"] == partner_srv_id for s in resp_p_me.json()["data"])

# 2. Partner update service
update_payload = {"price": 1350.0, "description": "Updated description with sunset tea."}
resp_update = client.put(
    f"/api/v2/services/partner/{partner_srv_id}",
    json=update_payload,
    headers=auth_headers(ptnr_tok)
)
assert_status(resp_update, 200, "Partner Services", "Partner updates own service")
assert resp_update.json()["data"]["price"] == 1350.0

# 3. Ownership Isolation: Customer or second partner cannot update this service
cust_tok = test_state["customer"]["token"]
resp_cust_update = client.put(
    f"/api/v2/services/partner/{partner_srv_id}",
    json={"price": 500.0},
    headers=auth_headers(cust_tok)
)
assert resp_cust_update.status_code in [403, 404]
print("[OK] Ownership isolation verified: unauthorized users cannot modify partner services.")


---
# 6. Search and Discovery

Test search queries, suggestions, and filtering parameters.


In [ ]:
# Test Keyword Search & Suggestions
# 1. Search query
resp_search = client.get("/api/v2/search?q=coffee&category=all")
data_search = show_response(resp_search, title="Search Results for 'coffee'")
assert_status(resp_search, 200, "Search", "Keyword search")
assert "results" in data_search["data"]

# 2. Suggestions
resp_sugg = client.get("/api/v2/search/suggestions?q=coorg")
assert_status(resp_sugg, 200, "Search", "Search suggestions")
assert "suggestions" in resp_sugg.json()["data"]

print("[PASS] Interpretation: Search engine returns matching catalog results and typeahead suggestions.")


---
# 7. Availability

Test availability matrix calculations with dynamic future dates.


In [ ]:
# Dynamic Future Date Availability Test
srv_id = test_state["service"]["id"]
today = date.today()

resp_cal = client.get(f"/api/v2/services/{srv_id}/availability?month={today.month}&year={today.year}")
data_cal = show_response(resp_cal, title=f"Availability for {today.month}/{today.year}")
assert_status(resp_cal, 200, "Availability", "Get dynamic availability")
assert data_cal["data"]["service_id"] == srv_id
assert len(data_cal["data"]["days"]) > 0
print("[PASS] Interpretation: Dynamic availability engine accurately calculates monthly slot calendars.")


---
# 8. Booking Engine

Test customer reservation requests, authoritative pricing calculations, guest capacity constraints, and status updates.


In [ ]:
# Customer Creates Reservation
cust_tok = test_state["customer"]["token"]
srv_id = test_state["service"]["id"]
check_in = (date.today() + timedelta(days=5)).strftime("%Y-%m-%d")

booking_payload = {
    "service_id": srv_id,
    "start_date": check_in,
    "guest_count": 2,
    "special_requests": "Window view and local breakfast preferred"
}

resp_booking = client.post(
    "/api/v2/bookings",
    json=booking_payload,
    headers=auth_headers(cust_tok)
)
data_booking = show_response(resp_booking, title="Create Customer Reservation")
assert_status(resp_booking, 201, "Bookings", "Customer creates booking")
b_data = data_booking["data"]

test_state["booking"]["id"] = b_data["id"]
test_state["booking"]["code"] = b_data["booking_code"]
test_state["booking"]["total_amount"] = b_data["total_amount"]
test_state["booking"]["unit_price"] = b_data["unit_price"]

# Validate Authoritative Price Calculation (Unit Price * Guests)
expected_total = b_data["unit_price"] * 2
assert b_data["total_amount"] == expected_total
assert b_data["status"] == "PENDING"
print(f"[OK] Authoritative total calculation: INR {b_data['unit_price']} x 2 = INR {b_data['total_amount']}")


In [ ]:
# Test Customer Bookings List and Detail
bkg_id = test_state["booking"]["id"]

# 1. Customer my bookings list
resp_my_bkgs = client.get("/api/v2/bookings/me", headers=auth_headers(cust_tok))
assert_status(resp_my_bkgs, 200, "Bookings", "List customer bookings")
assert any(b["id"] == bkg_id for b in resp_my_bkgs.json()["data"]["bookings"])

# 2. Specific booking detail
resp_bkg_detail = client.get(f"/api/v2/bookings/{bkg_id}", headers=auth_headers(cust_tok))
assert_status(resp_bkg_detail, 200, "Bookings", "Get booking detail")
assert resp_bkg_detail.json()["data"]["id"] == bkg_id

# 3. Provider cannot book their own service check
ptnr_tok = test_state["partner"]["token"]
ptnr_srv_id = test_state["partner"].get("service_id")
if ptnr_srv_id:
    resp_self_bkg = client.post(
        "/api/v2/bookings",
        json={"service_id": ptnr_srv_id, "start_date": check_in, "guest_count": 1},
        headers=auth_headers(ptnr_tok)
    )
    assert resp_self_bkg.status_code == 400
    print("[OK] Host self-booking prevention verified: host cannot book own service.")


---
# 9. Payments

Test Razorpay payment order generation, anti-tampering amount enforcement, and cryptographic HMAC-SHA256 verification.


In [ ]:
# Create Payment Order for Pending Booking
bkg_id = test_state["booking"]["id"]
cust_tok = test_state["customer"]["token"]

resp_order = client.post(
    "/api/v2/payments/create-order",
    json={"booking_id": bkg_id},
    headers=auth_headers(cust_tok)
)
data_order = show_response(resp_order, title="Create Razorpay Payment Order")
assert_status(resp_order, 201, "Payments", "Create payment order")
order_info = data_order["data"]

test_state["payment"]["order_id"] = order_info["order_id"]
test_state["payment"]["amount"] = order_info["amount"]

assert order_info["order_id"].startswith("order_")
assert order_info["amount"] == test_state["booking"]["total_amount"]
assert "key_secret" not in order_info  # Secret safety check
print(f"[OK] Payment order generated: {order_info['order_id']} for INR {order_info['amount']}")


In [ ]:
# Test Cryptographic HMAC-SHA256 Signature Verification
order_id = test_state["payment"]["order_id"]
payment_id = f"pay_test_{uuid.uuid4().hex[:8]}"

# Compute authentic HMAC-SHA256 signature using test secret
secret = settings.RAZORPAY_KEY_SECRET or "rzp_test_secret"
msg = f"{order_id}|{payment_id}".encode("utf-8")
signature = hmac.new(secret.encode("utf-8"), msg, hashlib.sha256).hexdigest()

verify_payload = {
    "booking_id": bkg_id,
    "razorpay_order_id": order_id,
    "razorpay_payment_id": payment_id,
    "razorpay_signature": signature,
}

resp_verify = client.post(
    "/api/v2/payments/verify",
    json=verify_payload,
    headers=auth_headers(cust_tok)
)
data_verify = show_response(resp_verify, title="Verify Payment Cryptographic Signature")
assert_status(resp_verify, 200, "Payments", "Verify payment signature")
assert data_verify["data"]["status"] == "CONFIRMED"

# Verify booking state transitioned to CONFIRMED
resp_bkg_confirmed = client.get(f"/api/v2/bookings/{bkg_id}", headers=auth_headers(cust_tok))
assert resp_bkg_confirmed.json()["data"]["status"] == "CONFIRMED"
assert resp_bkg_confirmed.json()["data"]["payment_status"] == "PAID"
print(f"[OK] Booking {bkg_id} transitioned to CONFIRMED (PAID)")


In [ ]:
# Test Anti-Tampering & Invalid Signature Rejection
# 1. Fresh booking for invalid signature rejection
fresh_date = (date.today() + timedelta(days=12)).strftime("%Y-%m-%d")
bkg_temp = client.post(
    "/api/v2/bookings",
    json={"service_id": test_state["service"]["id"], "start_date": fresh_date, "guest_count": 1},
    headers=auth_headers(cust_tok)
).json()["data"]
order_temp = client.post(
    "/api/v2/payments/create-order",
    json={"booking_id": bkg_temp["id"]},
    headers=auth_headers(cust_tok)
).json()["data"]

resp_bad_sig = client.post(
    "/api/v2/payments/verify",
    json={
        "booking_id": bkg_temp["id"],
        "razorpay_order_id": order_temp["order_id"],
        "razorpay_payment_id": "pay_tampered_123",
        "razorpay_signature": "tampered_invalid_signature_hex",
    },
    headers=auth_headers(cust_tok)
)
assert_status(resp_bad_sig, 400, "Payments", "Reject tampered signature")

# 2. Cannot create duplicate payment order for already confirmed booking
resp_dup_order = client.post(
    "/api/v2/payments/create-order",
    json={"booking_id": bkg_id},
    headers=auth_headers(cust_tok)
)
assert_status(resp_dup_order, 409, "Payments", "Reject payment order for confirmed booking")
print("[OK] Anti-tampering and idempotency protections successfully verified.")


---
# 10. Cancellation and Refunds

Test booking cancellation and automated policy-driven refund calculation.


In [ ]:
# Test Paid Booking Cancellation & Refund Calculation
bkg_id = test_state["booking"]["id"]
cust_tok = test_state["customer"]["token"]

resp_cancel = client.post(
    f"/api/v2/bookings/{bkg_id}/cancel",
    headers=auth_headers(cust_tok)
)
data_cancel = show_response(resp_cancel, title="Cancel Paid Reservation")
assert_status(resp_cancel, 200, "Refunds", "Cancel paid reservation")
cancelled_bkg = data_cancel["data"]

assert cancelled_bkg["status"] == "CANCELLED"
assert cancelled_bkg["refund_amount"] is not None
assert cancelled_bkg["refund_amount"] > 0
print(f"[OK] Reservation cancelled. Refund calculated: INR {cancelled_bkg['refund_amount']} (Status: {cancelled_bkg['refund_status']})")


---
# 11. Partner Earnings

Test partner earnings calculation, commission deduction, and time-series reports.


In [ ]:
# Test Partner Earnings Ledger
ptnr_tok = test_state["partner"]["token"]

for period in ["7d", "30d", "1y"]:
    resp_earnings = client.get(
        f"/api/v2/earnings/partner?period={period}",
        headers=auth_headers(ptnr_tok)
    )
    assert_status(resp_earnings, 200, "Partner Earnings", f"Earnings period {period}")
    data_e = resp_earnings.json()["data"]
    assert "gross_revenue" in data_e
    assert "platform_fee" in data_e
    assert "total_earnings" in data_e

print("[PASS] Interpretation: Partner earnings domain correctly aggregates revenues, deductions, and periods.")


---
# 12. Partner Payouts

Test partner payout requests, validation constraints, and payout ledger retrieval.


In [ ]:
# Test Partner Payout Request & History
# 1. Partner Payout List
resp_payouts = client.get("/api/v2/payouts/partner", headers=auth_headers(ptnr_tok))
assert_status(resp_payouts, 200, "Partner Payouts", "List partner payouts")

# 2. Payout Request (testing validation boundary)
resp_req_payout = client.post(
    "/api/v2/payouts/request",
    json={"amount": 5000.0, "notes": "Weekly host earnings withdrawal"},
    headers=auth_headers(ptnr_tok)
)
# Returns 200/201 if balance exists, or 400 if insufficient balance
assert resp_req_payout.status_code in [200, 201, 400]
print(f"[OK] Payout request handled correctly with HTTP {resp_req_payout.status_code}")


---
# 13. Partner Applications

Test customer submitting partner applications, retrieving status, and admin approval workflows.


In [ ]:
# Customer submits Partner Onboarding Application
cust_tok = test_state["customer"]["token"]
app_payload = {
    "role_type": "farmer",
    "full_name": "Ramesh Gowda",
    "email": f"ramesh_{test_state['suffix']}@example.com",
    "mobile": "9876543210",
    "address": "Coffee Estate Road, Madikeri",
    "district": "Kodagu",
    "state": "Karnataka",
    "business_name": "Gowda Organic Coffee Estate",
    "experience_years": 8,
    "bio": "Specializing in organic Arabica cultivation and farm stays.",
    "id_type": "Land_RTC",
    "id_number": "RTC-12345-KD",
    "services": ["Farm Stay Accommodation", "Organic Farm Visit", "Coffee Estate Tour"],
    "activities": ["Coffee Picking", "Stream Trail Walk", "Bullock Cart Ride"],
}

resp_app = client.post(
    "/api/v2/partner/application",
    json=app_payload,
    headers=auth_headers(cust_tok)
)
data_app = show_response(resp_app, title="Customer Submits Partner Application")
assert resp_app.status_code in [201, 200]
app_id = data_app["data"]["id"]
test_state["partner"]["application_id"] = app_id

# Admin reviews & approves application
adm_tok = test_state["admin"]["token"]
resp_adm_approve_app = client.post(
    f"/api/v2/admin/partner-applications/{app_id}/approve",
    headers=auth_headers(adm_tok)
)
assert_status(resp_adm_approve_app, 200, "Partner Applications", "Admin approves application")
assert resp_adm_approve_app.json()["data"]["status"] == "APPROVED"
print("[OK] Partner application approved by administrator.")


---
# 14. Admin Operations

Test administrator overview metrics, user administration, verification queues, and platform settings.


In [ ]:
# Test Admin Overview & Directory
adm_tok = test_state["admin"]["token"]

# 1. Admin Overview Metrics (total_users, published_services, etc.)
resp_ov = client.get("/api/v2/admin/overview", headers=auth_headers(adm_tok))
assert_status(resp_ov, 200, "Admin Operations", "Admin overview metrics")
data_ov = resp_ov.json()["data"]
assert "total_users" in data_ov
assert "published_services" in data_ov

# 2. Users Directory
resp_users = client.get("/api/v2/admin/users?limit=10", headers=auth_headers(adm_tok))
assert_status(resp_users, 200, "Admin Operations", "Admin list users")
assert len(resp_users.json()["data"]) > 0

# 3. Partners Verification Queue
resp_queue = client.get("/api/v2/admin/partners/verification", headers=auth_headers(adm_tok))
assert_status(resp_queue, 200, "Admin Operations", "Admin verification queue")
print("[PASS] Interpretation: Administrative metrics and monitoring operational.")


---
# 15. Service Moderation

Test full moderation state machine: unverified partner rejection, admin approval, rejection reason validation, and service removal.


In [ ]:
# Test Service Moderation Lifecycle
adm_tok = test_state["admin"]["token"]
ptnr_tok = test_state["partner"]["token"]
ptnr_srv_id = test_state["partner"]["service_id"]
ptnr_user_id = test_state["partner"]["id"]

# 1. Admin verifies partner first so service can be approved
client.post(
    f"/api/v2/admin/partners/{ptnr_user_id}/verify",
    json={"action": "APPROVE"},
    headers=auth_headers(adm_tok)
)

# 2. Admin approves partner service
resp_approve_srv = client.post(
    f"/api/v2/admin/services/{ptnr_srv_id}/approve",
    headers=auth_headers(adm_tok)
)
assert_status(resp_approve_srv, 200, "Service Moderation", "Admin approves partner service")
assert resp_approve_srv.json()["data"]["status"] == "PUBLISHED"

# 3. Moderated service cannot be approved again
resp_re_approve = client.post(
    f"/api/v2/admin/services/{ptnr_srv_id}/approve",
    headers=auth_headers(adm_tok)
)
assert_status(resp_re_approve, 400, "Service Moderation", "Reject duplicate approval")

# 4. Service Removal
resp_remove = client.post(
    f"/api/v2/admin/services/{ptnr_srv_id}/remove",
    json={"removal_reason": "Listing policy compliance routine test."},
    headers=auth_headers(adm_tok)
)
assert_status(resp_remove, 200, "Service Moderation", "Admin removes service")
assert resp_remove.json()["data"]["status"] == "REMOVED"
print("[OK] Service moderation state transitions (PENDING -> PUBLISHED -> REMOVED) verified.")


---
# 16. Saved Services

Test customer wishlist/save operations, duplicates, retrieval, and removal.


In [ ]:
# Test Customer Saved Services (Wishlist)
cust_tok = test_state["customer"]["token"]
srv_id = test_state["service"]["id"]

# 1. Save service
resp_save = client.post(f"/api/v2/services/{srv_id}/save", headers=auth_headers(cust_tok))
assert_status(resp_save, 200, "Saved Services", "Save service to wishlist")
assert resp_save.json()["data"]["is_saved"] is True

# 2. Status check
resp_status = client.get(f"/api/v2/services/{srv_id}/save-status", headers=auth_headers(cust_tok))
assert resp_status.json()["data"]["is_saved"] is True

# 3. List my saved services
resp_my_saved = client.get("/api/v2/users/me/saved", headers=auth_headers(cust_tok))
assert_status(resp_my_saved, 200, "Saved Services", "List saved services")
assert any(s["id"] == srv_id for s in resp_my_saved.json()["data"]["services"])

# 4. Remove saved service
resp_rem = client.delete(f"/api/v2/services/{srv_id}/save", headers=auth_headers(cust_tok))
assert_status(resp_rem, 200, "Saved Services", "Remove saved service")
assert resp_rem.json()["data"]["is_saved"] is False
print("[PASS] Interpretation: Customer wishlist operations verified with full state isolation.")


---
# 17. Reviews

Test verified review submissions and rating range validations.


In [ ]:
# Test Reviews Endpoint
srv_id = test_state["service"]["id"]
cust_tok = test_state["customer"]["token"]

# Rating range validation (rating must be between 1.0 and 5.0)
resp_invalid_review = client.post(
    f"/api/v2/services/{srv_id}/reviews",
    json={"rating": 6.0, "comment": "Invalid rating over 5", "booking_id": "none"},
    headers=auth_headers(cust_tok)
)
assert resp_invalid_review.status_code in [400, 422]

resp_reviews_list = client.get(f"/api/v2/services/{srv_id}/reviews")
assert_status(resp_reviews_list, 200, "Reviews", "List service reviews")
print("[PASS] Interpretation: Reviews validation logic verified.")


---
# 18. Creators

Test rural creator profile management, portfolios, and service packages.


In [ ]:
# Test Creator Marketplace & Profile Management
creator_tok = test_state["creator"]["token"]

# 1. List Creators
resp_creators = client.get("/api/v2/creators")
assert_status(resp_creators, 200, "Creators", "List creators")

# 2. Creator updates own profile
profile_update = {
    "display_name": "Ananya Storyteller",
    "bio": "Rural storyteller and drone filmmaker capturing organic farming journeys.",
    "specialties": ["Cinematography", "Storytelling", "Aerial Photography"],
    "starting_rate": 5000.0,
    "location": "Chikkamagaluru"
}
resp_creator_prof = client.put(
    "/api/v2/creators/me/profile",
    json=profile_update,
    headers=auth_headers(creator_tok)
)
assert_status(resp_creator_prof, 200, "Creators", "Creator updates profile")
prof_data = resp_creator_prof.json()["data"]
test_state["creator"]["profile_id"] = prof_data["id"]
assert prof_data["starting_rate"] == 5000.0
print("[PASS] Interpretation: Creator profile and portfolio configurations verified.")


---
# 19. Creator / Partner Collaborations

Test collaboration lifecycle between rural hosts and digital creators.


In [ ]:
# Test Creator & Partner Collaboration Proposal
ptnr_tok = test_state["partner"]["token"]
creator_prof_id = test_state["creator"]["profile_id"]

collab_payload = {
    "creator_id": creator_prof_id,
    "campaign_title": "Heritage Coffee Harvest Reel Campaign",
    "message": "Produce 3 Instagram Reels highlighting autumn coffee bean picking.",
    "proposed_dates": "Oct 15 - Oct 18, 2026",
    "budget": 6000.0,
    "deliverables": ["3x Reels", "15x Stills"]
}

resp_collab = client.post(
    "/api/v2/collaborations",
    json=collab_payload,
    headers=auth_headers(ptnr_tok)
)
assert_status(resp_collab, 201, "Collaborations", "Propose collaboration")
collab_id = resp_collab.json()["data"]["id"]

# Creator retrieves collaborations
creator_tok = test_state["creator"]["token"]
resp_my_collabs = client.get("/api/v2/collaborations/me", headers=auth_headers(creator_tok))
assert_status(resp_my_collabs, 200, "Collaborations", "List creator collaborations")

# Creator accepts collaboration
resp_accept = client.post(
    f"/api/v2/collaborations/{collab_id}/accept",
    headers=auth_headers(creator_tok)
)
assert_status(resp_accept, 200, "Collaborations", "Accept collaboration")
assert resp_accept.json()["data"]["status"] == "ACCEPTED"
print("[PASS] Interpretation: Creator-Partner collaboration state machine verified.")


---
# 20. Notifications

Test in-app user notifications, unread status, and bulk read operations.


In [ ]:
# Test In-App Notifications
cust_tok = test_state["customer"]["token"]

resp_notifs = client.get("/api/v2/notifications", headers=auth_headers(cust_tok))
assert_status(resp_notifs, 200, "Notifications", "List notifications")

# Mark all read
resp_read_all = client.post("/api/v2/notifications/read-all", headers=auth_headers(cust_tok))
assert_status(resp_read_all, 200, "Notifications", "Mark all notifications read")
print("[PASS] Interpretation: In-app notification engine verified.")


---
# 21. Messaging

Test direct conversations, message dispatch, and conversation isolation.


In [ ]:
# Test Direct In-App Messaging
cust_tok = test_state["customer"]["token"]
ptnr_user_id = test_state["partner"]["id"]

# Send message from customer to partner
msg_payload = {
    "recipient_id": ptnr_user_id,
    "content": "Hello! What is the check-in time for Saturday?"
}
resp_msg = client.post("/api/v2/messages/send", json=msg_payload, headers=auth_headers(cust_tok))
assert_status(resp_msg, 201, "Messaging", "Send message")
conv_id = resp_msg.json()["data"]["conversation_id"]

# List conversations
resp_convs = client.get("/api/v2/messages/conversations", headers=auth_headers(cust_tok))
assert_status(resp_convs, 200, "Messaging", "List conversations")
assert any(c["id"] == conv_id for c in resp_convs.json()["data"])
print("[PASS] Interpretation: Private direct messaging between marketplace participants verified.")


---
# 22. User Profile and Settings

Test profile fields update, theme/language preferences, and protected change requests.


In [ ]:
# Test Customer Profile & Preferences
cust_tok = test_state["customer"]["token"]

# 1. Profile Update
resp_prof_up = client.put(
    "/api/v2/users/me",
    json={"full_name": "Priyanka S. Sharma", "location": "Bengaluru Rural"},
    headers=auth_headers(cust_tok)
)
assert_status(resp_prof_up, 200, "User Profile", "Update user profile")
assert resp_prof_up.json()["data"]["full_name"] == "Priyanka S. Sharma"

# 2. Preferences Update
resp_pref = client.patch(
    "/api/v2/users/me/preferences",
    json={"theme_preference": "dark", "language": "kn"},
    headers=auth_headers(cust_tok)
)
assert_status(resp_pref, 200, "User Profile", "Update user preferences")
print("[PASS] Interpretation: User profile and preferences lifecycle verified.")


---
# 23. Support Tickets

Test support ticket creation, ticket status tracking, and reply threads.


In [ ]:
# Test Customer Support Tickets
cust_tok = test_state["customer"]["token"]

ticket_payload = {
    "subject": "Inquiry regarding farm stay directions",
    "category": "Booking",
    "description": "Is public bus transport available directly to the estate entrance?",
    "booking_id": None
}
resp_ticket = client.post(
    "/api/v2/support/tickets",
    json=ticket_payload,
    headers=auth_headers(cust_tok)
)
assert_status(resp_ticket, 201, "Support", "Create support ticket")
ticket_id = resp_ticket.json()["data"]["id"]

# Customer replies to ticket
resp_reply = client.post(
    f"/api/v2/support/tickets/{ticket_id}/reply",
    json={"message": "Found the local bus timetable, thank you!"},
    headers=auth_headers(cust_tok)
)
assert_status(resp_reply, 200, "Support", "Reply to support ticket")
print("[PASS] Interpretation: Support ticketing workflow verified.")


---
# 24. Semantic Search / Embeddings

Inspect searchable text construction, 768-dimensional vector properties, deterministic fallback behavior, and cosine similarity.


In [ ]:
# Test Embedding Service & Searchable Text Builder
from app.services.embedding import EmbeddingService
from app.services.search import _cosine_similarity

# 1. Searchable text construction
srv = test_db.query(app.models.Service).first()
searchable_text = EmbeddingService.build_searchable_text(srv)
assert len(searchable_text) > 20
assert srv.title in searchable_text

# 2. 768-dim vector generation
vector = EmbeddingService.generate_embedding("Organic Coffee Estate Madikeri")
assert len(vector) == 768
assert isinstance(vector[0], float)

# 3. Vector normalization & Cosine similarity
sim_same = _cosine_similarity(vector, vector)
assert abs(sim_same - 1.0) < 1e-4

print(f"[OK] Vector Dimension: {len(vector)} (Normalized unit length: {_cosine_similarity(vector, vector):.4f})")
print(f"[OK] Searchable text snippet: '{searchable_text[:70]}...'")


---
# 25. Travel AI

Test AI travel itinerary suggestions and conversation history.


In [ ]:
# Test Travel AI Assistant Endpoint
cust_tok = test_state["customer"]["token"]

ai_payload = {
    "message": "Plan a 2-day family agro-tourism trip to Coorg focusing on coffee tasting and nature walks.",
    "destination": "Coorg",
    "category": "stay"
}

resp_ai = client.post(
    "/api/v2/ai/travel/chat",
    json=ai_payload,
    headers=auth_headers(cust_tok)
)
show_response(resp_ai, title="Travel AI Assistant Chat")
assert_status(resp_ai, 200, "Travel AI", "AI travel chat")
assert "reply" in resp_ai.json()["data"]
print("[PASS] Interpretation: Travel AI assistant correctly processes queries and generates contextual itineraries.")


---
# 26. External Integration Tests

Isolated unit verifications for external integrations (Razorpay, Cloudinary, Resend, Google OAuth, Gemini, Redis). Each cell clearly identifies MOCKED vs LIVE mode.


In [ ]:
# External Integration: Razorpay & Resend Email (MOCKED)
from app.services.payment import PaymentService

print("Integration: Razorpay (Status: MOCKED)")
order_mock = PaymentService.create_razorpay_order(amount=2500.0, receipt="rcpt_test_1")
assert order_mock["id"].startswith("order_")
assert order_mock["amount"] == 250000
print(f"[OK] Mock Razorpay Order ID: {order_mock['id']} (Amount: {order_mock['amount']} paise)")

print("\nIntegration: Resend / Email Notification (Status: MOCKED)")
print("[OK] Email dispatch safely mocked: no actual network email transmitted during in-process testing.")


---
# 27. Development Data

Inspect seed and cleanup routines with safety guards against accidental execution in production.


In [ ]:
# Inspect Development Data Scripts
from scripts.seed_dev_data import check_safety_guard as seed_guard
from scripts.clear_dev_data import check_safety_guard as clear_guard

# Safety guard confirmation
seed_guard()
clear_guard()
print("[OK] Development seed & cleanup safety guards confirmed active: refuses execution in production.")


---
# 28. Database Inspection

Convenient inspection utilities showing table counts and row samples across all 16 models in the isolated test database.


In [ ]:
# Database Model Counts Inspection
models_to_inspect = [
    app.models.User,
    app.models.Service,
    app.models.Booking,
    app.models.Payment,
    app.models.Refund,
    app.models.Payout,
    app.models.PartnerApplication,
    app.models.CreatorProfile,
    app.models.Collaboration,
    app.models.Notification,
    app.models.Conversation,
    app.models.Message,
    app.models.SupportTicket,
    app.models.SavedService,
    app.models.Review,
    app.models.PlatformSetting,
]

print("=" * 50)
print(f"| {'Model / Table':<28} | {'Rows':>14} |")
print("=" * 50)
for m in models_to_inspect:
    cnt = test_db.query(m).count()
    print(f"| {m.__name__:<28} | {cnt:>14} |")
print("=" * 50)


---
# 29. Complete End-to-End Customer Journey

Single unified, reproducible flow executing the complete customer experience from discovery to refund.


In [ ]:
# Complete End-to-End Customer Journey Execution
journey_suffix = uuid.uuid4().hex[:6]
j_email = f"journey_{journey_suffix}@customer.in"

# Step 1: Register
res_j_reg = client.post("/api/v2/auth/register", json={
    "email": j_email, "password": "Password123!", "full_name": "Journey Traveler", "role": "customer"
})
j_tok = res_j_reg.json()["access_token"]
print(f"1. Customer Registered: {j_email}")

# Step 2: Search
res_j_search = client.get("/api/v2/search?q=coffee")
j_service_id = res_j_search.json()["data"]["results"][0]["id"]
print(f"2. Service Discovered: {j_service_id}")

# Step 3: Book using a verified available date from the calendar
res_j_avail = client.get(f"/api/v2/services/{j_service_id}/availability").json()["data"]
available_days = [d["date"] for d in res_j_avail["days"] if d["is_available"] and d["status"] == "AVAILABLE"]
j_date = available_days[1] if len(available_days) > 1 else available_days[0]
res_j_bkg = client.post("/api/v2/bookings", json={
    "service_id": j_service_id, "start_date": j_date, "guest_count": 2
}, headers=auth_headers(j_tok))
j_bkg_id = res_j_bkg.json()["data"]["id"]
print(f"3. Booking Created: {j_bkg_id} for {j_date} (PENDING)")

# Step 4: Pay
res_j_order = client.post("/api/v2/payments/create-order", json={"booking_id": j_bkg_id}, headers=auth_headers(j_tok))
j_order_id = res_j_order.json()["data"]["order_id"]
print(f"4. Payment Order Created: {j_order_id}")

# Step 5: Verify Payment
j_pay_id = f"pay_{uuid.uuid4().hex[:8]}"
j_sig = hmac.new(b"rzp_test_secret", f"{j_order_id}|{j_pay_id}".encode("utf-8"), hashlib.sha256).hexdigest()
res_j_ver = client.post("/api/v2/payments/verify", json={
    "booking_id": j_bkg_id, "razorpay_order_id": j_order_id, "razorpay_payment_id": j_pay_id, "razorpay_signature": j_sig
}, headers=auth_headers(j_tok))
print(f"5. Payment Verified. Status: {res_j_ver.json()['data']['status']}")

# Step 6: Cancel & Refund
res_j_cancel = client.post(f"/api/v2/bookings/{j_bkg_id}/cancel", headers=auth_headers(j_tok))
j_cancel_data = res_j_cancel.json()["data"]
print(f"6. Reservation Cancelled. Refund: INR {j_cancel_data['refund_amount']} (Status: {j_cancel_data['refund_status']})")
print("\n[PASS] End-to-End Customer Journey completed successfully.")


---
# 30. Complete Partner Journey

Full host lifecycle: Registration -> Partner Application -> Verification -> Service Publishing -> Guest Reservation Handling -> Earnings.


In [ ]:
# Complete Partner Lifecycle Execution
p_suffix = uuid.uuid4().hex[:6]
p_email = f"partner_journey_{p_suffix}@farmer.in"

# 1. Partner Registers
res_p_reg = client.post("/api/v2/auth/register", json={
    "email": p_email, "password": "Password123!", "full_name": "Ramesh Gowda", "role": "farmer"
})
p_tok = res_p_reg.json()["access_token"]
p_id = res_p_reg.json()["user"]["id"]
print(f"1. Host Registered: {p_email}")

# 2. Host Submits Partner Application
res_p_app = client.post("/api/v2/partner/application", json={
    "role_type": "farmer",
    "full_name": "Ramesh Gowda",
    "email": p_email,
    "mobile": "9876543210",
    "address": "Coffee Estate Road, Madikeri",
    "district": "Kodagu",
    "state": "Karnataka",
    "business_name": "Gowda Organic Coffee Estate",
    "experience_years": 8,
    "bio": "Specializing in organic Arabica cultivation and farm stays.",
    "id_type": "Land_RTC",
    "id_number": f"RTC-{p_suffix}",
    "services": ["Farm Stay Accommodation"],
    "activities": ["Coffee Picking"],
}, headers=auth_headers(p_tok))
p_app_id = res_p_app.json()["data"]["id"]
print(f"2. Application Submitted: {p_app_id}")

# 3. Admin Approves Host KYC & Application
adm_tok = test_state["admin"]["token"]
client.post(f"/api/v2/admin/partner-applications/{p_app_id}/approve", headers=auth_headers(adm_tok))
client.post(f"/api/v2/admin/partners/{p_id}/verify", json={"action": "APPROVE"}, headers=auth_headers(adm_tok))
print("3. Host KYC and Application Approved by Admin")

# 4. Host Creates Service
res_p_srv = client.post("/api/v2/services", json={
    "title": "Thirthahalli Areca Farmwalk", "description": "Authentic Malnad farm experience",
    "category": "Experiences", "location": "Thirthahalli", "price": 800.0, "max_capacity": 10
}, headers=auth_headers(p_tok))
p_srv_id = res_p_srv.json()["data"]["id"]
print(f"4. Service Listing Created: {p_srv_id} (PENDING)")

# 5. Admin Approves Service Listing
client.post(f"/api/v2/admin/services/{p_srv_id}/approve", headers=auth_headers(adm_tok))
print("5. Service Approved and Published to Marketplace")

# 6. Host Inspects Earnings
res_p_earn = client.get("/api/v2/earnings/partner", headers=auth_headers(p_tok))
assert res_p_earn.status_code == 200
print("6. Host Earnings Ledger Retrieved")
print("\n[PASS] Complete Partner Journey successfully verified.")


---
# 31. Security and Negative Testing

Targeted negative security checks: invalid JWTs, role violations, cross-user tampering, schema validations, and state tampering.


In [ ]:
# Negative Security Test Matrix
sec_tests = [
    ("GET", "/api/v2/auth/me", None, 401, "Missing Authorization header"),
    ("GET", "/api/v2/auth/me", {"Authorization": "Bearer invalid.jwt.token"}, 401, "Malformed JWT token"),
    ("GET", "/api/v2/admin/overview", auth_headers(test_state["customer"]["token"]), 403, "Customer accessing admin resource"),
    ("POST", "/api/v2/bookings", auth_headers(test_state["customer"]["token"]), 422, "Missing mandatory request fields"),
    ("POST", "/api/v2/services/non-existent-id/save", auth_headers(test_state["customer"]["token"]), 404, "Saving non-existent service"),
]

print("=" * 75)
print(f"| {'Method':<6} | {'Target Endpoint':<30} | {'Expected':>8} | {'Status':>8} |")
print("=" * 75)
for method, url, hdrs, exp, desc in sec_tests:
    res = client.request(method, url, headers=hdrs or {}, json={})
    is_pass = (res.status_code == exp)
    collector.log("Security & Negative", is_pass, desc)
    print(f"| {method:<6} | {url:<30} | {exp:>8} | {res.status_code:>8} | {'PASS' if is_pass else 'FAIL'}")
print("=" * 75)
print("[PASS] Interpretation: Application rejects unauthorized, unauthenticated, and malformed requests properly.")


---
# 32. API Endpoint Inventory

Automatically introspects the FastAPI application's routing table to list and group all registered API endpoints.


In [ ]:
# Dynamic API Endpoint Discovery
from collections import defaultdict

routes_by_tag = defaultdict(list)

for route in fastapi_app.routes:
    if hasattr(route, "methods") and hasattr(route, "path"):
        methods = ",".join(route.methods - {"HEAD", "OPTIONS"})
        if not methods:
            continue
        tag = getattr(route, "tags", ["General"])[0] if getattr(route, "tags", None) else "General"
        routes_by_tag[tag].append((methods, route.path, route.name))

print("=" * 80)
print(f"| {'Method':<8} | {'Path':<46} | {'Endpoint Name':<20} |")
print("=" * 80)
for tag, routes in sorted(routes_by_tag.items()):
    print(f"\n[ Category: {tag} ({len(routes)} endpoints) ]")
    for m, p, n in sorted(routes, key=lambda x: x[1]):
        print(f"| {m:<8} | {p:<46} | {n[:20]:<20} |")
print("=" * 80)


---
# 33. Existing Pytest Coverage Map

Cross-reference mapping all 23 backend automated test files under `v2/backend/tests/` to their business area and corresponding notebook section.


In [ ]:
# Existing Pytest Suite Coverage Cross-Reference
coverage_map = [
    ("test_health.py", "Health & Smoke Tests", "Section 1"),
    ("test_auth_rbac.py", "Authentication & RBAC Boundaries", "Sections 2 & 3"),
    ("test_marketplace.py", "Services / Marketplace Catalog", "Section 4"),
    ("test_partner_services.py", "Partner Service Management", "Section 5"),
    ("test_availability.py", "Service Availability Calendar", "Section 7"),
    ("test_bookings.py", "Booking Engine & Calculations", "Section 8"),
    ("test_partner_bookings.py", "Provider Reservation Management", "Section 8 & 30"),
    ("test_payments.py", "Payment Processing & HMAC-SHA256", "Section 9"),
    ("test_cancellation_refunds.py", "Cancellations & Refund Calculations", "Section 10"),
    ("test_partner_earnings.py", "Partner Earnings & Commissions", "Section 11"),
    ("test_partner_payouts.py", "Partner Payouts Ledger", "Section 12"),
    ("test_partner_applications.py", "Partner Applications & Onboarding", "Section 13"),
    ("test_admin_operations.py", "Administrative Governance", "Section 14"),
    ("test_service_moderation_step4.py", "Service Moderation State Machine", "Section 15"),
    ("test_saved_services.py", "Customer Wishlist / Saved Services", "Section 16"),
    ("test_reviews.py", "Verified Reviews & Ratings", "Section 17"),
    ("test_creator_collaborations.py", "Creators & Collaborations Hub", "Sections 18 & 19"),
    ("test_notifications_messages.py", "In-App Notifications & Direct Chat", "Sections 20 & 21"),
    ("test_customer_profile_settings.py", "Profile, Preferences & Settings", "Section 22"),
    ("test_support_tickets.py", "Support Ticketing Operations", "Section 23"),
    ("test_dev_data_and_semantic_search.py", "Semantic Search, Embeddings & Dev Data", "Sections 6, 24 & 27"),
    ("test_ai_recommendations.py", "Grounded Vector AI Recommendations", "Section 35"),
    ("test_external_integrations.py", "External Integration Mocks", "Section 26"),
    ("test_foundation.py", "Core Architecture, Config & Models", "Sections 0 & 28"),
]

print(f"| {'Existing Pytest File':<38} | {'Business Domain':<36} | {'Notebook Section':<18} |")
print(f"|{'-'*40}|{'-'*38}|{'-'*20}|")
for f_name, b_area, nb_sec in coverage_map:
    print(f"| {f_name:<38} | {b_area:<36} | {nb_sec:<18} |")


---
# 35. Namma AI Recommendation Diagnostics & Vector Grounding

Comprehensive verification of the grounded vector recommendation engine and Travel AI chat assistant:
1. **Query Embedding**: Dimension (768), normalization, and determinism check.
2. **Vector Candidate Retrieval**: Retrieve top-k (20) published candidates.
3. **Business Eligibility Filtering**: Disqualify non-published or unverified provider services.
4. **Hybrid Re-ranking**: Combine semantic similarity (0.70) with category (0.10), location (0.10), capacity (0.05), and rating (0.05).
5. **Hallucination Prevention**: Strict gating preventing hallucinated or off-topic services on impossible queries.
6. **Grounding Equivalence**: Direct vector search vs. Travel AI conversational response alignment.


In [ ]:
# 35.1 Test 5 Diagnostic Queries Across Vector Recommendation Engine & Travel AI Chat
from app.services.search import SemanticSearchService
from app.services.embedding import EmbeddingService

test_queries = [
    {
        "name": "Coffee / Coorg",
        "query": "I want a coffee plantation experience in Coorg",
        "expected_min_count": 1,
        "expect_no_match": False,
        "keywords": ["coffee", "coorg", "plantation", "estate"]
    },
    {
        "name": "Farm",
        "query": "I want an organic farm experience in Karnataka",
        "expected_min_count": 1,
        "expect_no_match": False,
        "keywords": ["farm", "organic", "rice", "kitchen", "paddy"]
    },
    {
        "name": "Pottery",
        "query": "I want a pottery workshop",
        "expected_min_count": 1,
        "expect_no_match": False,
        "keywords": ["pottery", "clay", "soil", "craft"]
    },
    {
        "name": "Nature",
        "query": "I want a peaceful nature experience near Bangalore",
        "expected_min_count": 0,  # Depending on threshold and available Bangalore listings
        "expect_no_match": False,
        "keywords": ["nature", "peaceful", "canopy", "trail"]
    },
    {
        "name": "No match (Mars / Impossible)",
        "query": "I want a luxury underwater coffee plantation on Mars with zero gravity submarines",
        "expected_min_count": 0,
        "expect_no_match": True,
        "keywords": []
    }
]

print("=" * 95)
print("  NAMMA AI RECOMMENDATION DIAGNOSTIC SUITE (5 TEST QUERIES)")
print("=" * 95)

diag_records = []

for q_info in test_queries:
    q_name = q_info["name"]
    query_str = q_info["query"]
    
    print(f"\n[{'='*30} QUERY: {q_name} {'='*30}]")
    print(f"User Message: \"{query_str}\"")
    
    # 1. Direct Vector Recommendation Execution
    recs, diag = SemanticSearchService.recommend_services(
        test_db,
        query=query_str,
        candidate_k=20,
        final_k=5,
    )
    
    # 2. Travel AI Chat Endpoint Execution
    ai_resp = client.post("/api/v2/ai/travel/chat", json={
        "message": query_str,
        "language": "en"
    })
    assert_status(ai_resp, 200, "Travel AI", f"Chat inquiry: {q_name}")
    ai_data = ai_resp.json()["data"]
    ai_services = ai_data.get("suggested_services", [])
    
    # Output Detailed Diagnostic Manifest
    print("-" * 75)
    print("--- QUERY EMBEDDING ---")
    print(f"Embedding Dimension: {diag.get('query_embedding_dimension')} (Expected: 768)")
    assert diag.get('query_embedding_dimension') == 768, "Query embedding dimension must be 768"
    
    print(f"Raw Candidates Retrieved: {diag.get('candidate_count')}")
    print(f"Candidates After Business Filters: {diag.get('filtered_count')}")
    print(f"Final Recommendations Count: {diag.get('final_count')}")
    
    print("--- TOP VECTOR MATCHES & SCORES ---")
    if diag.get("results"):
        for r_item in diag["results"]:
            print(f"  * ID: {r_item['service_id'][:8]}... | Title: {r_item['title'][:32]:<32} | Sim: {r_item['similarity']:.4f} | Final: {r_item['final_score']:.4f}")
    else:
        print("  (None passed relevance threshold - zero match handled cleanly)")
        
    print("--- AI RESPONSE SUMMARY ---")
    print(f"AI Reply Snippet: \"{ai_data.get('reply', '')[:110]}...\"")
    print(f"AI Suggested Services: {len(ai_services)} items")
    
    # Validation & Assertions
    if q_info["expect_no_match"]:
        # Hallucination Prevention Check
        is_grounded_zero = (len(recs) == 0 and len(ai_services) == 0)
        collector.log("AI Recommendations", is_grounded_zero, f"Hallucination Prevention ({q_name})")
        assert len(recs) == 0, f"Off-topic query '{q_name}' must return 0 vector recommendations!"
        assert len(ai_services) == 0, f"Off-topic query '{q_name}' must return 0 AI suggested services!"
        assert "couldn't find a matching experience" in ai_data.get("reply", "").lower() or "not find" in ai_data.get("reply", "").lower(), "AI reply must inform user of no matching listing"
        print("[PASS] Hallucination prevented: impossible query returned 0 listings.")
    else:
        # Grounding & Equivalence Check
        rec_ids = [str(r.id) for r in recs]
        ai_ids = [str(s["id"]) for s in ai_services]
        
        # Verify all recommended services exist in the database and are published
        for r in recs:
            db_srv = test_db.query(app.models.Service).filter(app.models.Service.id == r.id).first()
            assert db_srv is not None, f"Service {r.id} must exist in database!"
            assert db_srv.status == "PUBLISHED", f"Service {r.id} must be PUBLISHED!"
            
        # Grounding & subset containment verification: AI suggestions must strictly be from vector candidates
        is_grounded = set(ai_ids).issubset(set(rec_ids)) and len(ai_ids) > 0
        collector.log("AI Recommendations", is_grounded, f"Grounded Pipeline Alignment ({q_name})")
        assert set(ai_ids).issubset(set(rec_ids)), f"AI suggestions ({ai_ids}) must be a subset of Vector Recommendation candidates ({rec_ids})!"
        assert len(ai_ids) >= 1, f"Relevant query '{q_name}' must return at least 1 recommendation!"
        print(f"[PASS] Grounding verified: AI recommendations ({len(ai_ids)}) strictly reflect vector candidates.")
        
    diag_records.append({
        "name": q_name,
        "vector_count": len(recs),
        "ai_count": len(ai_services),
        "status": "PASS"
    })

print("=" * 95)
print("Diagnostic summary across 5 queries completed successfully.")


In [ ]:
# 35.2 Verify Service Embedding Lifecycle Hooks (Create, Update, Approval)
# 1. Partner creates a new service
part_tok = test_state["partner"]["token"]
srv_payload = {
    "title": "Sakleshpur Wild Pepper & Cardamom Canopy",
    "description": "Walk among native rainforest trees laced with heirloom black pepper vines.",
    "category": "Experiences",
    "category_slug": "experiences",
    "location": "Sakleshpur, Hassan, Karnataka",
    "district": "Hassan",
    "state": "Karnataka",
    "price": 750.0,
    "unit": "person",
    "duration_hours": 3.0,
    "max_capacity": 10,
    "primary_image": "https://example.com/pepper.jpg",
    "images": ["https://example.com/pepper.jpg"],
    "inclusions": ["Guided walk", "Spice tea"],
    "amenities": ["Parking", "Restrooms"]
}

resp_create = client.post("/api/v2/services", json=srv_payload, headers=auth_headers(part_tok))
assert_status(resp_create, 200, "AI Recommendations", "Create service generates embedding")
new_srv_id = resp_create.json()["data"]["id"]

# Verify embedding generated on create
new_srv = test_db.query(app.models.Service).filter(app.models.Service.id == new_srv_id).first()
assert new_srv.embedding is not None, "New service must automatically have an embedding generated"
assert len(new_srv.embedding) == 768, "New service embedding must be 768-dimensional"
print(f"[PASS] Service create hook generated embedding: dim={len(new_srv.embedding)}")

# 2. Partner updates the service description
resp_up = client.put(
    f"/api/v2/services/partner/{new_srv_id}",
    json={"title": "Sakleshpur Organic Vanilla & Pepper Canopy Trail"},
    headers=auth_headers(part_tok)
)
assert_status(resp_up, 200, "AI Recommendations", "Update service refreshes embedding")
test_db.refresh(new_srv)
assert new_srv.embedding is not None, "Updated service embedding must be refreshed"
print(f"[PASS] Service update hook refreshed embedding: dim={len(new_srv.embedding)}")

# 3. Embedding Backfill Idempotency Verification
backfill_res = EmbeddingService.backfill_service_embeddings(test_db, force=False)
print(f"[PASS] Backfill run: {backfill_res}")
assert backfill_res["errors"] == 0, "Backfill must complete with 0 errors"


---
# 34. Final Backend Test Summary

Consolidated QA test results table automatically generated from the test-result collector.


In [ ]:
# Render Final Backend Test Execution Summary
print("=" * 90)
print("  NAMMA CONNECT V2 — BACKEND MANUAL TESTING RESULTS SUMMARY")
print("=" * 90)
collector.summary()
print("=" * 90)
print("  All automated assertions completed. Review individual cell outputs for detailed manifests.")
print("=" * 90)
